# Produce the training data for a detector, with an encoder judge

A detector belongs to the model it scores, so training one for **your** model starts by
recording how that model answers: the questions, its answers *with the token
log-probabilities behind them*, and a verdict on each answer.

This notebook produces all three, against any OpenAI-compatible endpoint.

1. **Draw the questions** — from a QA dataset with short answers, or your own pack.
2. **Answer them**, keeping `top_logprobs`. The distribution behind the answer is what the
   detector reads; the answer text is only used to judge it.
3. **Judge the answers** against their gold answers, with **BERT-as-a-Judge** — a 210M
   encoder that runs here, not at the endpoint.
4. **Fit and evaluate** a WEPR detector on the result, and save it.

It is the sibling of [train_wepr_pipeline](train_wepr_pipeline.ipynb), identical but for step 3, so the two can
be diffed against each other. That step is the whole difference, and it buys two things:

**Judging stops costing anything.** The LLM judge is a second request per answer, so a run
is 2N requests; here it is N. The judge downloads once (~420 MB) and then runs locally, on
CPU if there is no GPU.

**Labels become reproducible.** An encoder at inference is deterministic, so re-running
step 3 on the same answers gives the same targets. A generative judge at `temperature=0`
is *nearly* deterministic and not quite, which is a poor property for the labels a
classifier is fitted on.

What it gives up: an explanation. The LLM judge says why in prose; this one returns a
number. When a verdict looks wrong, there is nothing to read — only a score to compare
against the threshold.

## What it writes

Three files, joined on `custom_id`:

| File | Contents |
|---|---|
| `questions.json` | The question pack: question, id, gold answer, aliases |
| `responses.jsonl` | The answers, with `top_logprobs` per token |
| `scores.jsonl` | One correctness score per answer, and the label it becomes |

`responses.jsonl` is the **OpenAI Batch output shape** — one JSON object per line wrapping
a chat completion under `custom_id` — because that is what the endpoint returned, so it is
readable by anything that reads a batch.

`scores.jsonl` is not, and deliberately so. An encoder judge produces a probability, not a
chat completion; dressing one up as an API response would put a reply in the file that no
API ever sent. It is a flat record instead — `custom_id`, `score`, `threshold`,
`hallucination` — which is the honest shape and which step 4 reads directly. The cost is
that `scripts/train_detector.py` and [train_wepr](train_wepr.ipynb) read Batch-shaped verdicts, so
feeding them this run means a conversion; the last section shows it.

Writing the files out at all is the point. Generating costs money and time; fitting costs
seconds. With them on disk you can refit at a different `k`, on `epr` instead of `wepr`, or
against a different threshold, without paying for any of it twice.

## Prerequisites

```bash
uv pip install "artefactual[adapters]" datasets torch "transformers>=4.57"
```

| Variable | Required | What it is |
|---|---|---|
| `OPENAI_BASE_URL` | yes | Any OpenAI-compatible endpoint returning `top_logprobs` |
| `OPENAI_API_KEY` | yes | Its key |
| `OPENAI_MODEL` | yes | The model being scored — the detector you train belongs to it, and the id has to be one your endpoint serves |

The endpoint is needed for step 2 only. Step 3 reaches the Hugging Face Hub once, for the
judge's weights, and nothing after that leaves the machine.

**The endpoint has to return at least `K` ranks per token**, which is the one requirement
worth checking before you start. OpenAI's own API accepts `top_logprobs` up to 20, and a
self-hosted vLLM up to its `--max-logprobs` (20 by default), so `K = 15` fits both.
Providers that cap lower, or omit `logprobs` entirely, are refused by name in step 2 rather
than silently training on narrower data.

**A detector belongs to the model it was trained on.** Its weights read that model's
confidence, so scoring a different model with them is not supported; retrain instead.

This notebook is not executed when the documentation is built, because it generates against
a live endpoint. The numbers you see are the ones your run produces.

In [ ]:
# Colab, or any other kernel that is not this repository's environment: install what this
# notebook needs.
# A checkout that ran `uv sync --extra adapters` has all of it but torch, which nothing
# else here needs and which is therefore in no dependency group.
#
# uv rather than pip, because pip is the slow half of the wait: installing this package
# into an empty environment measured 17 s under pip, against 3 s to pip-install uv plus 1 s
# for uv to do the same work. On Colab, where most of the dependency tree is already
# present, the gap is smaller.
#
# Plain Python rather than the `!pip` and `%pip` magics, so the cell stays valid Python:
# the tests that run these notebooks compile the code cells, and so do the linters.
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path


def missing(distribution):
    """Whether `distribution` is installed in the interpreter running this kernel.

    Asked of the installed distribution rather than of an import, because a bare directory
    named `artefactual` -- which is what cloning this repository beside the notebook leaves
    behind -- is an empty namespace package: `find_spec` finds it and `import artefactual`
    succeeds, so both would report the package present and skip the install. Only the
    metadata distinguishes a directory from a package.
    """
    try:
        importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return True
    return False


def shadowed(name):
    """Whether a directory beside the notebook hides an installed package of that name.

    The working directory comes first on `sys.path`, so such a directory wins over
    anything installed and the import fails on a submodule, several cells from the cause.
    """
    return Path(name).is_dir() and not Path(name, "__init__.py").exists()


def install(*packages):
    """Install into the interpreter running this kernel, showing what went wrong if it does.

    `-q` and no captured output is how an install failure becomes a bare
    `CalledProcessError` with the resolver's explanation nowhere on screen.
    """
    bootstrap = subprocess.run([sys.executable, "-m", "pip", "install", "-qU", "uv"], capture_output=True, text=True)
    resolve = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, "-q", *packages]
    done = bootstrap if bootstrap.returncode else subprocess.run(resolve, capture_output=True, text=True)
    if done.returncode:
        print(done.stderr or done.stdout)
        done.check_returncode()
    # The kernel started before these files existed, so the import machinery has a cached
    # listing of a directory that did not contain them.
    importlib.invalidate_caches()


if missing("artefactual") or missing("openai") or missing("datasets") or missing("torch") or missing("transformers"):
    install("artefactual[adapters]", "datasets", "torch", "transformers>=4.57")

assert not shadowed("artefactual"), (
    "a directory named 'artefactual' beside this notebook is hiding the installed "
    "package; rename it, or run this notebook from somewhere else"
)

In [ ]:
import contextlib
import json
import os
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# No default: a model id only means something to the endpoint serving it, and a wrong one
# fails on every request in step 2 rather than here.
MODEL = os.environ["OPENAI_MODEL"]

# Ranks kept per token. Part of the feature definition, not a batch size: WEPR fits one
# coefficient per rank, so a detector is only ever used at the k it was fitted at. Every
# published detector uses 15.
K = 15
# The API caps it at 20, and an endpoint asked for more rejects every request in step 2 --
# which arrives as "no answers were generated" three cells later, pointing at the wrong
# thing.
assert 1 <= K <= 20, "top_logprobs must be between 1 and 20"

# Start small. This is 2 requests per question -- one to answer, one to judge -- so 25
# questions is 50 requests and a minute or two. It is a smoke test, not a training run: at
# this size a capable model may hallucinate two or three times, which is too few to split
# and too few to score. Raise it once a run has come back clean; a few hundred is where the
# numbers start to mean something.
N_QUESTIONS = 25
SEED = 42
WORKERS = 8

# The judge. `artefactory/BERTJudge` is the checkpoint its model card recommends, and is
# the paper's `BERTJudge-Free-QCR`. "Free" is the half that matters here: it was trained on
# unconstrained generations, which is what step 2 produces, where the `Formatted` siblings
# expect an answer ending in "Final answer: <x>". QCR is the input it reads -- question,
# candidate, reference; `artefactory/BERTJudge-Free-CR` is the same judge without the
# question, for packs where the question adds nothing.
JUDGE_MODEL = "artefactory/BERTJudge"

# P(correct) at or above which the answer counts as correct. 0.5 is the midpoint of a
# sigmoid trained against a binary target, and the paper's published example scores land at
# 0.02-0.99, nowhere near it. It is still the single number that decides every label, so
# step 3 prints the distribution and how close to it the run came.
THRESHOLD = 0.5

# Sequences per forward pass. Raise it on a GPU; 8 keeps the memory flat on a laptop.
JUDGE_BATCH = 8

QUESTIONS = Path("questions.json")
RESPONSES = Path("responses.jsonl")
SCORES = Path("scores.jsonl")

## Step 1 — build the question pack

Four fields per question. `question_id` is the one that travels: it becomes `custom_id` on
the answers and the verdicts, and that is what every later join pairs on.

TriviaQA's closed-book configuration (`rc.nocontext`) carries all four, so the mapping is a
rename. **Any short-form QA set works** — only these column names change. Two properties
decide whether one is usable, and neither is about its columns: answers must be **short
enough for a judge to grade** against `short_answer`, and the model must get **enough of
them wrong** that both classes appear. A model that answers everything correctly leaves the
fit nothing to learn from.

Shuffled before slicing, because splits arrive grouped by source and the head of one is a
narrower sample than the same count drawn at random.

Written to disk rather than kept in memory, so a rerun of the later steps answers the
same questions -- and so a pack you already have can be dropped in instead of this cell.

In [ ]:
from datasets import load_dataset

rows = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")

questions = [
    {
        "question": row["question"],
        "question_id": row["question_id"],
        "short_answer": row["answer"]["value"],
        # `value` goes in its own field, so the aliases that merely restate it are dropped
        # rather than repeated to the judge.
        "answer_aliases": [a for a in row["answer"]["aliases"] if a.casefold() != row["answer"]["value"].casefold()],
    }
    for row in rows.shuffle(seed=SEED).select(range(N_QUESTIONS))
]

# Duplicate ids would collapse in the join, pairing an answer with another question's gold
# answer -- a wrong label rather than an error, so it is refused here.
assert len({q["question_id"] for q in questions}) == len(questions), "question_id is not unique"

QUESTIONS.write_text(json.dumps(questions, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"wrote {QUESTIONS}, {len(questions)} questions")
print(json.dumps(questions[0], indent=2)[:400])

## Step 2 — generate the answers, keeping the log-probabilities

`logprobs=True` and `top_logprobs=K` are what make a response scoreable at all: the
detector reads the token distribution behind the answer, not the answer. A response
generated without them is a valid completion carrying nothing to score, and one generated
with fewer than `K` ranks is refused when it reaches the parser rather than zero-filled —
the missing ranks are unfetched rather than absent, so filling them with zeros would score
the answer as more confident than it was. Generating *wider* than `K` is safe; surplus
ranks are dropped.

Prompt and sampling follow the paper (§4.1.2): non-greedy at `T = 1.0`, `top_p = 1.0`.
The paper also sets `top_k = 50`, which is not an OpenAI parameter — so this samples at
whatever the endpoint's default is, and the distribution is not quite the paper's. It also
sends `max_completion_tokens`, which some OpenAI-compatible servers still only accept as
`max_tokens`; if every request fails, that is the first thing to check.
Non-greedy is the point — the method measures hesitation in the raw distribution.

Requests are threaded because the round trips, not the fitting, are what make this slow. A
failure returns `None` and is written out as an error row, exactly as a batch job records
one, so the count is visible rather than silently missing.

In [ ]:
from openai import OpenAI, OpenAIError

# `max_retries` above the SDK's default of 2: this fires N requests at once, and a burst
# of 429s that exhausts the retries becomes a thinner dataset rather than an error.
client = OpenAI(max_retries=6)  # reads OPENAI_BASE_URL and OPENAI_API_KEY

# Which endpoint this is actually talking to. Unset, OPENAI_BASE_URL silently means
# api.openai.com, and a self-hosted run then fails N times with an authentication error.
print(f"endpoint: {client.base_url}")

GENERATE = """You are a useful assistant that help finding short and precise answers for a given query or question.
            Please keep your output AS SHORT AND CONCISE AS POSSIBLE.
            Here is the query :
            {query}
            """


def generate(question):
    try:
        return client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": GENERATE.format(query=question["question"])}],
            logprobs=True,
            top_logprobs=K,
            temperature=1.0,
            top_p=1.0,
            max_completion_tokens=200,
        )
    except OpenAIError as error:
        # Every failure this call can produce -- transport, timeout, rate limit, a
        # rejected request -- is an OpenAIError, and one of them should not cost the
        # run. Anything else is a bug in the code above and should not be caught here.
        return error


with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    generated = list(pool.map(generate, questions))  # `map` preserves order

# The OpenAI Batch output shape: one line per request, `custom_id` carrying the question id
# and the ChatCompletion in `response.body`. Failures become error rows rather than gaps.
with RESPONSES.open("w", encoding="utf-8") as out:
    for question, result in zip(questions, generated):
        failed = isinstance(result, Exception)
        out.write(
            json.dumps(
                {
                    "id": f"chatcmpl-{question['question_id']}",
                    "custom_id": question["question_id"],
                    "response": None if failed else {"status_code": 200, "body": result.model_dump()},
                    # The class name, not the provider's text: an authentication error
                    # quotes the key it rejected, and this file is one you hand onward.
                    # The full message is printed below, where it stays in the session.
                    "error": {"message": type(result).__name__} if failed else None,
                },
                # ASCII-escaped, unlike questions.json: every reader of these files splits
                # them with `splitlines()`, which breaks on U+2028, U+2029 and U+0085 --
                # characters JSON does not require escaping and a model can emit.
                ensure_ascii=True,
            )
            + "\n"
        )

ok = [(q, r) for q, r in zip(questions, generated) if not isinstance(r, Exception)]
print(f"wrote {RESPONSES}, {len(ok)}/{len(generated)} generated")
for question, result in zip(questions, generated):
    if isinstance(result, Exception):
        print(f"  failed: {question['question_id']}: {result}")

In [ ]:
# Nothing came back at all -- almost always OPENAI_BASE_URL, the key, or a model name the
# endpoint does not serve. Checked before indexing, because the errors above say what
# happened and a bare IndexError here would not.
assert ok, (
    "no answers were generated. Check OPENAI_BASE_URL, OPENAI_API_KEY and OPENAI_MODEL, "
    "and read the per-request errors above: an endpoint that rejects `top_logprobs`, "
    "`temperature` or `max_completion_tokens` fails every request the same way."
)

# The cheapest place to notice an endpoint that ignored `top_logprobs`: it returns a
# perfectly valid completion carrying nothing to score. `logprobs` is checked before
# `.content` because that is the shape the failure actually takes -- reaching straight for
# `.content` raises an AttributeError that names nothing useful.
# `.content` is optional inside `logprobs` as well, and a provider that returns the
# object with nothing in it fails the same way for the reader.
missing = [q["question_id"] for q, r in ok if not (r.choices[0].logprobs and r.choices[0].logprobs.content)]
assert not missing, (
    f"{len(missing)} response(s) carry no logprobs at all, e.g. {missing[:3]}. "
    f"The endpoint accepted `logprobs=True` and ignored it; it cannot be used at any k."
)

# Every token, not just the first -- a response whose later tokens are narrower would pass
# a first-token check and fail inside `fit`.
widths = [len(t.top_logprobs) for _, r in ok for t in (r.choices[0].logprobs.content or [])]
assert widths and min(widths) >= K, (
    f"endpoint returned {min(widths) if widths else 0} ranks per token, need {K}; "
    f"raise top_logprobs, or lower K and fit at that rank count"
)

print(f"{min(widths)}-{max(widths)} ranks per token across {len(ok)} answers")
print(f"{ok[0][0]['question']}\n  -> {ok[0][1].choices[0].message.content.strip()}")

## Step 3 — judge the answers

[BERT-as-a-Judge](https://arxiv.org/abs/2604.09497) scores answer correctness from a
question, a candidate answer and a reference answer, without an LLM in the loop. The paper
is an evaluation paper — its target is benchmark scoring — and labelling hallucinations is
the same question asked once per row: *is this answer right, given the gold one?*

The input is three marker tokens and the three fields, with no instructions, no JSON and no
prompt to render:

```
<|question|>{question}<|candidate|>{answer}<|reference|>{gold}
```

That is the whole interface, which is why it is inlined below rather than imported. The
model returns two logits, and the score is the margin of *correct* over *incorrect* pushed
through a sigmoid — one probability per answer.

**One reference, not the alias list.** The judge takes a single reference string, so it
sees `short_answer` and not the aliases the LLM judge is shown. On TriviaQA that is the
interesting difference between the two notebooks: the aliases are exactly where a lexical
match fails and a semantic judge should not need them. If your gold answers have several
equally-canonical forms, score against each and keep the best — one line, at the end of
this step.

**The judge answers the opposite question to the one we record.** Its score is P(*correct*);
the file we write states `hallucination`, the positive class the detector predicts. The
conversion is `hallucination = score < THRESHOLD`, done once, at the point of writing.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# The judge's input, built exactly as the reference implementation builds it
# (github.com/artefactory/BERT-as-a-Judge, `src/bert_judge/judges/bert.py::_make_prompts`).
# The three markers are already in this checkpoint's vocabulary -- ids 128256 to 128258 of
# 128259 -- so nothing is added to the tokenizer here and no embedding row is left
# randomly initialised, which is what makes reproducing this in plain transformers safe.
QUESTION_MARKER = "<|question|>"
CANDIDATE_MARKER = "<|candidate|>"
REFERENCE_MARKER = "<|reference|>"


def judge_input(question, candidate, reference):
    """The one string the judge scores."""
    return f"{QUESTION_MARKER}{question}{CANDIDATE_MARKER}{candidate}{REFERENCE_MARKER}{reference}"


tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL, trust_remote_code=True)
# Left, so an over-long input loses the start of the question rather than the reference at
# the end -- which is the half the verdict depends on.
tokenizer.truncation_side = "left"

device = "cuda" if torch.cuda.is_available() else "cpu"
judge_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        JUDGE_MODEL,
        num_labels=2,
        trust_remote_code=True,
        # bfloat16 is the checkpoint's own dtype and what the reference implementation
        # loads; on CPU it is slower than float32 for a model this size.
        dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    )
    .to(device)
    .eval()
)

print(f"{JUDGE_MODEL} on {device}, {sum(p.numel() for p in judge_model.parameters()) / 1e6:.0f}M parameters")

In [ ]:
@torch.no_grad()
def correctness(prompts, batch_size=JUDGE_BATCH):
    """P(correct) for each prompt, reduced as the reference implementation reduces it."""
    scored = []
    for start in range(0, len(prompts), batch_size):
        batch = tokenizer(
            prompts[start : start + batch_size],
            padding=True,
            truncation=True,
            max_length=judge_model.config.max_position_embeddings,
            return_tensors="pt",
        ).to(device)
        logits = judge_model(**batch).logits
        # Two logits into one probability: the margin of `correct` over `incorrect`. Read
        # off column 1 minus column 0, not a softmax, because that is what the judge was
        # trained to produce and what its published scores are.
        scored += torch.sigmoid(logits[:, 1] - logits[:, 0]).float().tolist()
    return scored


answers = [completion.choices[0].message.content.strip() for _, completion in ok]
prompts = [
    judge_input(question["question"], answer, question["short_answer"]) for (question, _), answer in zip(ok, answers)
]
scores = correctness(prompts)

# A flat record per answer, not a chat completion: the judge is an encoder and never
# produced one. `score` is kept beside the label so a different threshold can be applied
# later without running the judge again -- which is most of the point of writing a file.
with SCORES.open("w", encoding="utf-8") as out:
    for (question, _), answer, score in zip(ok, answers, scores):
        out.write(
            json.dumps(
                {
                    "custom_id": question["question_id"],
                    "judge": JUDGE_MODEL,
                    "score": round(score, 6),
                    "threshold": THRESHOLD,
                    # The one place the judge's convention and the detector's meet:
                    # the judge scores `correct`, the detector predicts `hallucination`.
                    "hallucination": score < THRESHOLD,
                    "candidate": answer,
                    "reference": question["short_answer"],
                },
                ensure_ascii=True,
            )
            + "\n"
        )

judgments = [(question, completion, score < THRESHOLD) for (question, completion), score in zip(ok, scores)]

print(f"wrote {SCORES}, {len(scores)} scored answers")

# How far the run sat from the threshold. A judge that is sure looks like the paper's
# published example -- scores at the ends, nothing in the middle. A pile of scores near
# THRESHOLD means the labels are decided by the cut rather than by the judge, and the
# threshold is then worth choosing rather than accepting.
undecided = [score for score in scores if abs(score - THRESHOLD) < 0.1]
print(f"  min {min(scores):.3f}, max {max(scores):.3f}, {len(undecided)} within 0.1 of THRESHOLD={THRESHOLD}")
for question, completion, flag in judgments[:2]:
    said = completion.choices[0].message.content.strip()
    print(f"  [{'hallucination' if flag else 'grounded'}] said {said[:40]!r} (gold: {question['short_answer']!r})")

In [ ]:
import numpy as np

y = np.array([flag for _, _, flag in judgments], dtype=int)

# Both classes are needed, and this is where a run fails cheaply rather than inside `fit`.
# All-correct means the questions were too easy for this model; all-wrong usually means it
# is not answering in the short form the judge expects, or that THRESHOLD is wrong.
assert 0 < y.sum() < len(y), f"labels are single-class ({y.sum()}/{len(y)}); the questions need to be harder or easier"

print(f"{len(y)} scored answers, {y.sum()} hallucinations ({y.mean():.0%})")

## Step 4 — fit and evaluate

The three files are written, so from here everything is repeatable for free. This step
reads them back rather than using what is still in memory -- which is what makes it true
that you can come back tomorrow, change `k`, and refit without paying for generation
again.

`trainable=True` returns an unfitted pipeline -- parser, entropy reduction, logistic
regression -- that takes the raw responses, so there is no feature extraction to write.

**ROC-AUC** scores the ranking, which governs triage by score and is what the paper
reports; the **classification report** scores the decisions at 0.5, where recall on the
`hallucination` row is the fraction actually flagged. Only the AUC carries over to another
threshold.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from artefactual.scoring import wepr


# Read back from disk, not from the variables above: this is the path a fresh kernel takes,
# and the one anything else reading these files takes too.
def completions(path):
    """Every usable completion in a Batch output file, by `custom_id`.

    `.get` and the blank-line skip because this reads a file, not the variables above: it
    is the same code that would read a file written by anything else.
    """
    found = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        envelope = row.get("response") or {}
        if row.get("error") is not None or not envelope:
            continue
        found[row["custom_id"]] = envelope.get("body", envelope)
    return found


def verdicts(path):
    """Every score in the judge's output file, by `custom_id`.

    The label is recomputed from `score` and `THRESHOLD` rather than read from
    `hallucination`, so changing the threshold above and re-running this cell relabels the
    run without re-scoring it. The stored flag is what the file states; this is what the
    fit uses, and the assert holds the two together.
    """
    found = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        row = json.loads(line)
        found[row["custom_id"]] = float(row["score"])
    return found


answers = completions(RESPONSES)
scored = verdicts(SCORES)
labelled = [(answers[custom_id], int(score < THRESHOLD)) for custom_id, score in scored.items() if custom_id in answers]

missing = len(scored) - len(labelled)
if missing:
    print(f"{missing} score(s) name an answer that is not in {RESPONSES.name} and are not trained on")

responses = [completion for completion, _ in labelled]
y = np.array([label for _, label in labelled])
print(f"read {len(responses)} labelled answers back from {RESPONSES.name} and {SCORES.name}")

# Enough of the rarer class to sit on both sides of the split and mean something once
# there. At N_QUESTIONS = 25 a capable model can land under this; that is the smoke test
# telling you it was a smoke test.
rarer = min(y.sum(), len(y) - y.sum())
assert rarer >= 5, (
    f"only {rarer} answer(s) in the rarer class out of {len(y)}; a holdout cannot say "
    f"anything at that size. Raise N_QUESTIONS and rerun steps 1-3."
)

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)
detector = wepr(k=K, trainable=True).fit(x_train, y_train)
print(f"fitted on {len(y_train)}, holding out {len(y_test)}")

scores = detector.predict_proba(x_test)[:, 1]
print(f"\nROC-AUC: {roc_auc_score(y_test, scores):.2f}")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

In [ ]:
path = detector.save_estimator("wepr-bertjudge.skops")
print(f"wrote {path}")

reloaded = wepr(path, k=K)

# Held-out answers: the rows the fit above never saw. `responses` holds the completions as
# they were read back from the file, so they are plain dicts rather than SDK objects.
for completion, label in list(zip(x_test, y_test))[:5]:
    probability = reloaded.predict_proba(completion)[0, 1]
    said = completion["choices"][0]["message"]["content"].strip()
    print(f"[{'hallucination' if label else 'grounded    '}] P={probability:.3f}  said {said[:40]!r}")

## Where to go next

- **Choose the threshold rather than accept it.** `scores.jsonl` keeps the raw score, so
  moving `THRESHOLD` and re-running step 4 relabels the run in seconds, with no judge and
  no endpoint. If the distribution printed in step 3 had a pile near the cut, this is the
  knob that decided those labels.
- **Compare the two judges.** Run [train_wepr_pipeline](train_wepr_pipeline.ipynb) over the same
  `responses.jsonl` and join the two verdict files on `custom_id`. Where they disagree is
  where the alias list mattered, or where one of them is wrong — and on a labelling task
  that is worth looking at before trusting either.
- **Feed the CLI, or the training notebook.** Both read the Batch-shaped verdicts an LLM
  judge writes, which this file is not. The conversion is explicit rather than hidden,
  because it invents an envelope no API produced:

  ```python
  with Path("judgments.jsonl").open("w", encoding="utf-8") as out:
      for line in SCORES.read_text(encoding="utf-8").splitlines():
          row = json.loads(line)
          verdict = json.dumps({"judgment": not row["hallucination"], "explanation": f"BERTJudge score {row['score']}"})
          body = {"choices": [{"index": 0, "message": {"role": "assistant", "content": verdict}}]}
          out.write(json.dumps({"id": f"bertjudge-{row['custom_id']}", "custom_id": row["custom_id"],
                                "response": {"status_code": 200, "body": body}, "error": None}) + "\n")
  ```

- **More questions.** The generation is the cost and both the judge and the fit are
  seconds, so raise `N_QUESTIONS` rather than economising on labels.
- **Another dataset.** Only step 1 changes. Training data should resemble the traffic being
  scored: the paper's numbers drop 10-20 points when a TriviaQA-trained detector meets
  WebQuestions.
- **Another judge checkpoint.** `artefactory/BERTJudge-Free-CR` drops the question from the
  input, and the `Formatted` family expects answers that end in `Final answer: <x>`, which
  is not the shape step 2 asks for. The paper's own recommendation is the one set above.